In [1]:
# Clearing the environment

try:
    del full_dataset, train_indices, val_indices, train_subset, val_subset, train_loader, val_loader, labels, class_names, model, criterion, optimizer
except NameError:
    print("Some are non-existent!")

!rm -rf /kaggle/working/*
%reset -f


import gc

def reset_memory():
    # Delete all variables.
    for var in gc.get_objects():
        try:
            del var
        except:
            pass

    # Run the garbage collector.
    gc.collect()

    print("Memory has been reset")

# Call the function to reset memory

reset_memory()

threshold = 0.4

Some are non-existent!
Memory has been reset


# Preparing dataset

In [2]:
import json
import numpy as np
from PIL import Image
from skimage.draw import polygon

def load_mask_from_json(json_path, text_path, height, width):
    with open(json_path) as f:
        json_data = json.load(f)

    with open(text_path) as g:
        labels = []
        for line in g:
            labels.append(int(line.strip()[0]) + 1)

    masks = []
    boxes = []

    for shape in json_data["shapes"]:
        pts = np.array(shape["points"], dtype=np.int32)
        rr, cc = polygon(pts[:,1], pts[:,0], (height, width))
        
        mask = np.zeros((height, width), dtype=np.uint8)
        mask[rr, cc] = 1
        masks.append(mask)

        # bounding box [x_min, y_min, x_max, y_max]
        x_min, y_min = pts[:,0].min(), pts[:,1].min()
        x_max, y_max = pts[:,0].max(), pts[:,1].max()
        boxes.append([x_min, y_min, x_max, y_max])

    return {
        "boxes": np.array(boxes, dtype=np.float32),
        "labels": np.array(labels, dtype=np.int64),
        "masks": np.stack(masks, axis=0)
    }

label_map = {
    1: {"name": "G-cocci", "color": "red", "color_code": "r"},
    2: {"name": "G+cocci", "color": "green", "color_code": "g"},
    3: {"name": "G-bacilli", "color": "blue", "color_code": "b"},
    4: {"name": "G+bacilli", "color": "cyan", "color_code": "c"}
}

In [3]:
import torch
from torch.utils.data import Dataset
import os
from PIL import Image
from torchvision import transforms
import torch.nn.functional as F

torch.cuda.empty_cache()

class BacteriaDataset(Dataset):
    def __init__(self, img_dir, json_dir, text_dir, transforms=None):
        self.img_dir = img_dir
        self.json_dir = json_dir
        self.text_dir = text_dir
        self.transforms = transforms
        self.imgs = sorted(os.listdir(img_dir))
        self.json = sorted(os.listdir(json_dir))
        self.text = sorted(os.listdir(text_dir))

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.imgs[idx])
        json_path = os.path.join(self.json_dir, self.json[idx])
        text_path = os.path.join(self.text_dir, self.text[idx])

        img = Image.open(img_path).convert("RGB")
        w, h = img.size
        target = load_mask_from_json(json_path, text_path, h, w)

        # convert numpy → torch tensors
        target = {k: torch.as_tensor(v) for k, v in target.items()}
        target["image_id"] = torch.tensor([idx])

        new_h, new_w = (224, 224)

        scale_y = new_h / h
        scale_x = new_w / w

        # resize masks
        if "masks" in target:
            target["masks"] = F.interpolate(
                target["masks"].unsqueeze(1).float(),  # add channel dimension
                size=(new_h, new_w),
                mode="nearest"  # for binary masks, NEVER use bilinear
            ).squeeze(1)
    
        # resize boxes
        if "boxes" in target:
            boxes = target["boxes"]
            boxes[:, [0, 2]] *= scale_x
            boxes[:, [1, 3]] *= scale_y
            target["boxes"] = boxes

        # only apply transform to image
        if self.transforms:
            img = self.transforms(img)

        return img, target

    def __len__(self):
        return len(self.imgs)

# fold management

In [15]:
import torchvision
from torch.utils.data import DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split
from torch.utils.data import Subset
import numpy as np

transform = transforms.Compose([
    transforms.Resize((224, 224)), 
    transforms.ToTensor(), 
    transforms.Normalize([0.5]*3, [0.5]*3)
])

dataset = BacteriaDataset("/kaggle/input/m-rose-dataset-segmentation-and-detection/640DataSet/images/", 
                          "/kaggle/input/m-rose-dataset-segmentation-and-detection/640DataSet/json/", 
                          "/kaggle/input/m-rose-dataset-segmentation-and-detection/640DataSet/labels/", transforms=transform)

def get_fold_indices(dataset_size, fold_idx, n_folds=5):
    
    all_indices = np.arange(dataset_size)
    
    # Calculate the size of each fold
    fold_size = dataset_size // n_folds
    
    # Calculate start and end indices for test set
    val_start = (fold_idx - 1) * fold_size
    
    # For the last fold, include any remaining samples
    if fold_idx == n_folds:
        val_end = dataset_size
    else:
        val_end = fold_idx * fold_size
    
    # Get test indices
    val_idx = all_indices[val_start:val_end]
    
    # Get train indices: all indices except test indices
    train_idx = np.concatenate([all_indices[:val_start], all_indices[val_end:]])
    
    return train_idx, val_idx

def collate_fn(batch):
    return tuple(zip(*batch))

# Example usage for a specific fold
fold_idx = 5

train_idx, val_idx = get_fold_indices(len(dataset), fold_idx)

train_dataset = Subset(dataset, train_idx)
val_dataset = Subset(dataset, val_idx)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=4, collate_fn=collate_fn)

val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=4, collate_fn=collate_fn)

# initilizing model

In [5]:
from torchvision.models.detection import MaskRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.transform import GeneralizedRCNNTransform

num_classes = 5  # 4 classes + background
fixed_transform = GeneralizedRCNNTransform(
    min_size=224,
    max_size=224,
    image_mean=[0.5]*3,
    image_std=[0.5]*3
)

model = torchvision.models.detection.maskrcnn_resnet50_fpn(weights=MaskRCNN_ResNet50_FPN_Weights.DEFAULT)
model.transform = fixed_transform
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = torchvision.models.detection.faster_rcnn.FastRCNNPredictor(in_features, num_classes)
in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
hidden_layer = 256
model.roi_heads.mask_predictor = torchvision.models.detection.mask_rcnn.MaskRCNNPredictor(in_features_mask, hidden_layer, num_classes)

# Training loop
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)

print(f"Total Parameters = {sum(p.numel() for p in model.parameters())/1e6} M")
print(f"Trainable Parameters = {sum(p.numel() for p in model.parameters())/1e6} M")

optimizer = torch.optim.AdamW(model.parameters(), lr = 1e-4, betas = (0.9, 0.99), weight_decay = 0.001)

Downloading: "https://download.pytorch.org/models/maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth" to /root/.cache/torch/hub/checkpoints/maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth


100%|██████████| 170M/170M [00:00<00:00, 222MB/s] 


Total Parameters = 43.938541 M
Trainable Parameters = 43.938541 M


# metrics during training

In [6]:
import torch
from tqdm import tqdm
import numpy as np
from collections import defaultdict

def compute_instance_metrics_classwise(preds, targets, num_classes, iou_thresh=threshold, smooth=1e-6):
    """
    preds: list of dicts from Mask R-CNN (after inference)
    targets: list of dicts with "masks" and "labels"
    num_classes: total number of object classes (excluding background)
    """
    class_stats = {cls: {"tp": 0, "fp": 0, "fn": 0, "ious": []} for cls in range(1, num_classes)}

    for pred, tgt in zip(preds, targets):
        pred_masks = (pred["masks"] > 0.5).squeeze(1).cpu().numpy()
        pred_labels = pred["labels"].cpu().numpy()
        tgt_masks = tgt["masks"].cpu().numpy()
        tgt_labels = tgt["labels"].cpu().numpy()

        for cls in range(1, num_classes):
            pmasks = [m for m, l in zip(pred_masks, pred_labels) if l == cls]
            tmasks = [m for m, l in zip(tgt_masks, tgt_labels) if l == cls]
            matched_gt = set()

            for pm in pmasks:
                best_iou = 0.0
                best_idx = -1
                for i, tm in enumerate(tmasks):
                    inter = np.logical_and(pm, tm).sum()
                    union = np.logical_or(pm, tm).sum()
                    iou = (inter + smooth) / (union + smooth)
                    if iou > best_iou:
                        best_iou = iou
                        best_idx = i
                if best_iou >= iou_thresh:
                    class_stats[cls]["tp"] += 1
                    class_stats[cls]["ious"].append(best_iou)
                    matched_gt.add(best_idx)
                else:
                    class_stats[cls]["fp"] += 1

            # count false negatives
            class_stats[cls]["fn"] += len(tmasks) - len(matched_gt)

    # --- Compute metrics per class ---
    results = {}
    for cls in range(1, num_classes):
        tp = class_stats[cls]["tp"]
        fp = class_stats[cls]["fp"]
        fn = class_stats[cls]["fn"]
        ious = class_stats[cls]["ious"]

        precision = tp / (tp + fp + smooth)
        recall = tp / (tp + fn + smooth)
        mean_iou = np.mean(ious) if ious else 0.0
        dice = (2 * tp + smooth) / (2 * tp + fp + fn + smooth)
        acc = tp / (tp + fp + fn + smooth)

        results[f"precision_class_{cls}"] = precision
        results[f"recall_class_{cls}"] = recall
        results[f"iou_class_{cls}"] = mean_iou
        results[f"dice_class_{cls}"] = dice
        results[f"accuracy_class_{cls}"] = acc

    # --- Mean (macro) metrics ---
    results["mean_precision"] = np.mean([results[f"precision_class_{c}"] for c in range(1, num_classes)])
    results["mean_recall"]    = np.mean([results[f"recall_class_{c}"] for c in range(1, num_classes)])
    results["mean_iou"]       = np.mean([results[f"iou_class_{c}"] for c in range(1, num_classes)])
    results["mean_dice"]      = np.mean([results[f"dice_class_{c}"] for c in range(1, num_classes)])
    results["mean_accuracy"]  = np.mean([results[f"accuracy_class_{c}"] for c in range(1, num_classes)])

    return results

# model training

In [ ]:
best_dice = 0.0
best_model_path = "best_instance_segmentation.pth"

epochs = 25
train_losses, val_losses = [], []
train_history, val_history = [] , []

for epoch in range(epochs):
    # === Training ===
    model.train()
    running_loss = 0.0

    for images, targets in train_loader:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)   # training mode
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        running_loss += losses.item()

    train_loss = running_loss / len(train_loader)
    train_losses.append(train_loss)

    # === Validation (inference mode) ===
    model.eval()
    val_metrics_list = []
    val_running_loss = 0.0
    with torch.no_grad():
        for images, targets in val_loader:
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            # forward pass in training mode to get loss
            model.train()  # switch to training mode for loss computation
            loss_dict = model(images, targets)
            val_loss = sum(loss for loss in loss_dict.values())
            val_running_loss += val_loss.item()

            # inference mode for predictions
            # --- back to eval mode for predictions ---
            model.eval()
            outputs = model(images)
            metrics = compute_instance_metrics_classwise(outputs, targets, num_classes=num_classes, iou_thresh=threshold)
            val_metrics_list.append(metrics)

    val_loss = val_running_loss / len(val_loader)
    val_losses.append(val_loss)

    # average metrics
    val_metrics = {k: np.mean([m[k] for m in val_metrics_list]) for k in val_metrics_list[0].keys()}
    val_metrics["loss"] = val_loss
    val_metrics["dice_loss"] = 1 - val_metrics["mean_dice"]
    val_metrics["iou_loss"] = 1 - val_metrics["mean_iou"]

    train_history.append({"loss": train_loss})
    val_history.append(val_metrics)

    # === Logging ===
    print(f"Epoch {epoch+1}/{epochs} | "
          f"Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f} || "
          f"Dice: {val_metrics['mean_dice']:.4f}, Dice Loss: {val_metrics['dice_loss']:.4f} | "
          f"IoU: {val_metrics['mean_iou']:.4f}, IoU Loss: {val_metrics['iou_loss']:.4f} | "
          f"Prec: {val_metrics['mean_precision']:.4f}, Recall: {val_metrics['mean_recall']:.4f}, Accuracy: {val_metrics['mean_accuracy']:.4f}")

    # === Save best model ===
    if val_metrics["mean_dice"] > best_dice:
        best_dice = val_metrics["mean_dice"]
        torch.save(model.state_dict(), best_model_path)
        print(f"  -> Saved new best model (Dice={best_dice:.4f})")

In [ ]:
# Device and model setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model.load_state_dict(torch.load("best_instance_segmentation.pth", map_location=device))
model.to(device)
model.eval()
all_preds, all_targets = [], []

with torch.no_grad():
    for images, targets in tqdm(val_loader, desc="Evaluating"):
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        # --- Run model inference ---
        outputs = model(images)

        # --- Collect predictions and ground truths ---
        # Move everything to CPU and detach to avoid GPU memory buildup
        for out, tgt in zip(outputs, targets):
            pred = {
                "boxes":  out["boxes"].cpu(),
                "labels": out["labels"].cpu(),
                "scores": out["scores"].cpu(),
                "masks":  out["masks"].cpu(),  # shape: (N_pred, 1, H, W)
            }
            gt = {
                "boxes":  tgt["boxes"].cpu(),
                "labels": tgt["labels"].cpu(),
                "masks":  tgt["masks"].cpu(),  # shape: (N_gt, H, W)
            }
            all_preds.append(pred)
            all_targets.append(gt)

final_metrics = compute_instance_metrics_classwise(all_preds, all_targets, num_classes=num_classes)

for cls in range(1, num_classes):
    print(f"Class {cls}:")
    print(f"  Dice:      {final_metrics[f'dice_class_{cls}']:.4f}")
    print(f"  IoU:       {final_metrics[f'iou_class_{cls}']:.4f}")
    print(f"  Precision: {final_metrics[f'precision_class_{cls}']:.4f}")
    print(f"  Recall:    {final_metrics[f'recall_class_{cls}']:.4f}")
    print(f"  Accuracy:  {final_metrics[f'accuracy_class_{cls}']:.4f}")

print("\nOverall Mean:")
print(f"  Mean Dice: {final_metrics['mean_dice']:.4f}")
print(f"  Mean IoU:  {final_metrics['mean_iou']:.4f}")
print(f"  Mean Precision: {final_metrics['mean_precision']:.4f}")
print(f"  Mean Recall:    {final_metrics['mean_recall']:.4f}")

# calculate all metrics

In [16]:
import torch
import numpy as np
from collections import defaultdict

# Device and model setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"CUDA is available. Using GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("CUDA is not available. Using CPU.")

model.load_state_dict(torch.load("/kaggle/input/models/aliakbarratul/mask-rcnn-inst-seg/pytorch/default/1/best_mask_rcnn_seg_fold_5.pth", map_location=device))
model.to(device)
model.eval()

def compute_detection_metrics(preds, targets, num_classes, iou_thresh=threshold):
    """
    Compute object detection metrics: Precision, Recall, F1, mAP
    
    Args:
        preds: list of dicts with 'boxes', 'labels', 'scores'
        targets: list of dicts with 'boxes', 'labels'
        num_classes: number of classes (excluding background)
        iou_thresh: IoU threshold for matching boxes
    """
    class_stats = {cls: {"tp": 0, "fp": 0, "fn": 0, "scores": [], "matched": []} 
                   for cls in range(1, num_classes)}
    
    for pred, tgt in zip(preds, targets):
        pred_boxes = pred["boxes"].cpu().numpy()
        pred_labels = pred["labels"].cpu().numpy()
        pred_scores = pred["scores"].cpu().numpy()
        
        tgt_boxes = tgt["boxes"].cpu().numpy()
        tgt_labels = tgt["labels"].cpu().numpy()
        
        for cls in range(1, num_classes):
            # Filter predictions and targets for this class
            pred_idx = pred_labels == cls
            tgt_idx = tgt_labels == cls
            
            cls_pred_boxes = pred_boxes[pred_idx]
            cls_pred_scores = pred_scores[pred_idx]
            cls_tgt_boxes = tgt_boxes[tgt_idx]
            
            matched_gt = set()
            
            # Sort predictions by score (highest first)
            if len(cls_pred_scores) > 0:
                sorted_idx = np.argsort(-cls_pred_scores)
                cls_pred_boxes = cls_pred_boxes[sorted_idx]
                cls_pred_scores = cls_pred_scores[sorted_idx]
            
            # Match predictions to ground truth
            for pb, ps in zip(cls_pred_boxes, cls_pred_scores):
                best_iou = 0.0
                best_idx = -1
                
                for i, tb in enumerate(cls_tgt_boxes):
                    if i in matched_gt:
                        continue
                    iou = box_iou(pb, tb)
                    if iou > best_iou:
                        best_iou = iou
                        best_idx = i
                
                if best_iou >= iou_thresh and best_idx != -1:
                    class_stats[cls]["tp"] += 1
                    class_stats[cls]["matched"].append(1)
                    matched_gt.add(best_idx)
                else:
                    class_stats[cls]["fp"] += 1
                    class_stats[cls]["matched"].append(0)
                
                class_stats[cls]["scores"].append(ps)
            
            # Count false negatives
            class_stats[cls]["fn"] += len(cls_tgt_boxes) - len(matched_gt)
    
    # Compute metrics per class
    results = {}
    all_precisions = []
    all_recalls = []
    all_f1s = []
    all_aps = []
    
    for cls in range(1, num_classes):
        tp = class_stats[cls]["tp"]
        fp = class_stats[cls]["fp"]
        fn = class_stats[cls]["fn"]
        scores = class_stats[cls]["scores"]
        matched = class_stats[cls]["matched"]
        
        # Precision, Recall, F1
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
        
        # Average Precision (AP)
        ap = compute_ap(scores, matched, fn)
        
        results[f"det_precision_class_{cls}"] = precision
        results[f"det_recall_class_{cls}"] = recall
        results[f"det_f1_class_{cls}"] = f1
        results[f"det_ap_class_{cls}"] = ap
        
        all_precisions.append(precision)
        all_recalls.append(recall)
        all_f1s.append(f1)
        all_aps.append(ap)
    
    # Mean metrics
    results["det_mean_precision"] = np.mean(all_precisions)
    results["det_mean_recall"] = np.mean(all_recalls)
    results["det_mean_f1"] = np.mean(all_f1s)
    results["det_mAP"] = np.mean(all_aps)
    
    return results


def compute_classification_metrics(preds, targets, num_classes):
    """
    Compute classification metrics: Accuracy, Precision, Recall, F1
    Treats each detected object as a classification instance
    """
    class_stats = {cls: {"tp": 0, "fp": 0, "fn": 0, "tn": 0} 
                   for cls in range(1, num_classes)}
    
    for pred, tgt in zip(preds, targets):
        pred_boxes = pred["boxes"].cpu().numpy()
        pred_labels = pred["labels"].cpu().numpy()
        
        tgt_boxes = tgt["boxes"].cpu().numpy()
        tgt_labels = tgt["labels"].cpu().numpy()
        
        # Match predictions to ground truth based on IoU
        matched_gt = set()
        matched_pred = set()
        
        for i, pb in enumerate(pred_boxes):
            best_iou = 0.0
            best_idx = -1
            
            for j, tb in enumerate(tgt_boxes):
                if j in matched_gt:
                    continue
                iou = box_iou(pb, tb)
                if iou > best_iou:
                    best_iou = iou
                    best_idx = j
            
            if best_iou >= 0.5 and best_idx != -1:
                matched_gt.add(best_idx)
                matched_pred.add(i)
                
                pred_cls = pred_labels[i]
                true_cls = tgt_labels[best_idx]
                
                # Update stats for all classes
                for cls in range(1, num_classes):
                    if pred_cls == cls and true_cls == cls:
                        class_stats[cls]["tp"] += 1
                    elif pred_cls == cls and true_cls != cls:
                        class_stats[cls]["fp"] += 1
                    elif pred_cls != cls and true_cls == cls:
                        class_stats[cls]["fn"] += 1
                    else:
                        class_stats[cls]["tn"] += 1
        
        # Unmatched predictions are false positives
        for i in range(len(pred_labels)):
            if i not in matched_pred:
                pred_cls = pred_labels[i]
                class_stats[pred_cls]["fp"] += 1
        
        # Unmatched ground truths are false negatives
        for j in range(len(tgt_labels)):
            if j not in matched_gt:
                true_cls = tgt_labels[j]
                class_stats[true_cls]["fn"] += 1
    
    # Compute metrics per class
    results = {}
    all_accuracies = []
    all_precisions = []
    all_recalls = []
    all_f1s = []
    
    for cls in range(1, num_classes):
        tp = class_stats[cls]["tp"]
        fp = class_stats[cls]["fp"]
        fn = class_stats[cls]["fn"]
        tn = class_stats[cls]["tn"]
        
        accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0.0
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
        
        results[f"clf_accuracy_class_{cls}"] = accuracy
        results[f"clf_precision_class_{cls}"] = precision
        results[f"clf_recall_class_{cls}"] = recall
        results[f"clf_f1_class_{cls}"] = f1
        
        all_accuracies.append(accuracy)
        all_precisions.append(precision)
        all_recalls.append(recall)
        all_f1s.append(f1)
    
    # Mean metrics
    results["clf_mean_accuracy"] = np.mean(all_accuracies)
    results["clf_mean_precision"] = np.mean(all_precisions)
    results["clf_mean_recall"] = np.mean(all_recalls)
    results["clf_mean_f1"] = np.mean(all_f1s)
    
    return results


def box_iou(box1, box2):
    """
    Compute IoU between two boxes [x1, y1, x2, y2]
    """
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - intersection
    
    return intersection / union if union > 0 else 0.0


def compute_ap(scores, matched, fn):
    """
    Compute Average Precision given scores and match status
    """
    if len(scores) == 0:
        return 0.0
    
    # Sort by score
    sorted_idx = np.argsort(-np.array(scores))
    matched = np.array(matched)[sorted_idx]
    
    # Compute precision-recall curve
    tp = np.cumsum(matched)
    fp = np.cumsum(1 - matched)
    
    recalls = tp / (tp[-1] + fn) if (tp[-1] + fn) > 0 else tp * 0
    precisions = tp / (tp + fp)
    
    # Compute AP using 11-point interpolation
    ap = 0.0
    for t in np.linspace(0, 1, 11):
        if np.sum(recalls >= t) == 0:
            p = 0
        else:
            p = np.max(precisions[recalls >= t])
        ap += p / 11
    
    return ap

# ============================================================
# USAGE EXAMPLE
# ============================================================

def evaluate_all_metrics(model, val_loader, device, num_classes):
    """
    Comprehensive evaluation: Detection, Classification, and Segmentation metrics
    """
    model.eval()
    all_preds, all_targets = [], []
    
    with torch.no_grad():
        for images, targets in val_loader:
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            
            outputs = model(images)
            
            for out, tgt in zip(outputs, targets):
                pred = {
                    "boxes": out["boxes"].cpu(),
                    "labels": out["labels"].cpu(),
                    "scores": out["scores"].cpu(),
                    "masks": out["masks"].cpu(),
                }
                gt = {
                    "boxes": tgt["boxes"].cpu(),
                    "labels": tgt["labels"].cpu(),
                    "masks": tgt["masks"].cpu(),
                }
                all_preds.append(pred)
                all_targets.append(gt)
    
    # Compute all metrics
    det_metrics = compute_detection_metrics(all_preds, all_targets, num_classes)
    clf_metrics = compute_classification_metrics(all_preds, all_targets, num_classes)
    # seg_metrics = compute_instance_metrics_classwise(all_preds, all_targets, num_classes)
    
    # Combine all metrics
    all_metrics = {**det_metrics, **clf_metrics}  # , **seg_metrics}
    
    return all_metrics, all_preds, all_targets


def print_all_metrics(metrics, num_classes):
    """
    Print all metrics in organized format
    """
    print("="*80)
    print("OBJECT DETECTION METRICS")
    print("="*80)
    for cls in range(1, num_classes):
        print(f"\nClass {cls}:")
        print(f"  Precision: {metrics[f'det_precision_class_{cls}']:.4f}")
        print(f"  Recall:    {metrics[f'det_recall_class_{cls}']:.4f}")
        print(f"  F1 Score:  {metrics[f'det_f1_class_{cls}']:.4f}")
        print(f"  AP:        {metrics[f'det_ap_class_{cls}']:.4f}")
    
    print(f"\nOverall Detection:")
    print(f"  Mean Precision: {metrics['det_mean_precision']:.4f}")
    print(f"  Mean Recall:    {metrics['det_mean_recall']:.4f}")
    print(f"  Mean F1:        {metrics['det_mean_f1']:.4f}")
    print(f"  mAP:            {metrics['det_mAP']:.4f}")
    
    print("\n" + "="*80)
    print("CLASSIFICATION METRICS")
    print("="*80)
    for cls in range(1, num_classes):
        print(f"\nClass {cls}:")
        print(f"  Accuracy:  {metrics[f'clf_accuracy_class_{cls}']:.4f}")
        print(f"  Precision: {metrics[f'clf_precision_class_{cls}']:.4f}")
        print(f"  Recall:    {metrics[f'clf_recall_class_{cls}']:.4f}")
        print(f"  F1 Score:  {metrics[f'clf_f1_class_{cls}']:.4f}")
    
    print(f"\nOverall Classification:")
    print(f"  Mean Accuracy:  {metrics['clf_mean_accuracy']:.4f}")
    print(f"  Mean Precision: {metrics['clf_mean_precision']:.4f}")
    print(f"  Mean Recall:    {metrics['clf_mean_recall']:.4f}")
    print(f"  Mean F1:        {metrics['clf_mean_f1']:.4f}")
    
    # If segmentation metrics are included
    if 'mean_dice' in metrics:
        print("\n" + "="*80)
        print("SEGMENTATION METRICS")
        print("="*80)
        for cls in range(1, num_classes):
            print(f"\nClass {cls}:")
            print(f"  Dice:      {metrics[f'dice_class_{cls}']:.4f}")
            print(f"  IoU:       {metrics[f'iou_class_{cls}']:.4f}")
            print(f"  Precision: {metrics[f'precision_class_{cls}']:.4f}")
            print(f"  Recall:    {metrics[f'recall_class_{cls}']:.4f}")
        
        print(f"\nOverall Segmentation:")
        print(f"  Mean Dice: {metrics['mean_dice']:.4f}")
        print(f"  Mean IoU:  {metrics['mean_iou']:.4f}")


# Collect all predictions and targets
all_preds, all_targets = [], []

with torch.no_grad():
    for images, targets in tqdm(val_loader, desc="Evaluating"):
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        
        outputs = model(images)
        
        for out, tgt in zip(outputs, targets):
            pred = {
                "boxes": out["boxes"].cpu(),
                "labels": out["labels"].cpu(),
                "scores": out["scores"].cpu(),
                "masks": out["masks"].cpu(),
            }
            gt = {
                "boxes": tgt["boxes"].cpu(),
                "labels": tgt["labels"].cpu(),
                "masks": tgt["masks"].cpu(),
            }
            all_preds.append(pred)
            all_targets.append(gt)

# Compute all metrics
print("\nComputing Detection Metrics...")
det_metrics = compute_detection_metrics(all_preds, all_targets, num_classes)

print("Computing Classification Metrics...")
clf_metrics = compute_classification_metrics(all_preds, all_targets, num_classes)

print("Computing Segmentation Metrics...")
seg_metrics = compute_instance_metrics_classwise(all_preds, all_targets, num_classes)

# Combine all metrics
all_metrics = {**det_metrics, **clf_metrics, **seg_metrics}

# Print comprehensive results
print_all_metrics(all_metrics, num_classes)

CUDA is available. Using GPU: Tesla P100-PCIE-16GB


Evaluating: 100%|██████████| 76/76 [00:29<00:00,  2.54it/s]



Computing Detection Metrics...
Computing Classification Metrics...
Computing Segmentation Metrics...
OBJECT DETECTION METRICS

Class 1:
  Precision: 0.4422
  Recall:    0.9044
  F1 Score:  0.5939
  AP:        0.7175

Class 2:
  Precision: 0.7333
  Recall:    0.6598
  F1 Score:  0.6946
  AP:        0.5613

Class 3:
  Precision: 0.5466
  Recall:    0.8620
  F1 Score:  0.6690
  AP:        0.7110

Class 4:
  Precision: 0.5607
  Recall:    0.7235
  F1 Score:  0.6318
  AP:        0.5255

Overall Detection:
  Mean Precision: 0.5707
  Mean Recall:    0.7874
  Mean F1:        0.6473
  mAP:            0.6288

CLASSIFICATION METRICS

Class 1:
  Accuracy:  0.6001
  Precision: 0.4099
  Recall:    0.8385
  F1 Score:  0.5506

Class 2:
  Accuracy:  0.6319
  Precision: 0.5154
  Recall:    0.4637
  F1 Score:  0.4882

Class 3:
  Accuracy:  0.8174
  Precision: 0.4577
  Recall:    0.7217
  F1 Score:  0.5601

Class 4:
  Accuracy:  0.8872
  Precision: 0.3714
  Recall:    0.4793
  F1 Score:  0.4185

Overall 

# related plots

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm

# Device and model setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"CUDA is available. Using GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("CUDA is not available. Using CPU.")

# model.load_state_dict(torch.load("/kaggle/working/best_instance_segmentation.pth", map_location=device))
model.to(device)
model.eval()

def box_iou(box1, box2):
    """Compute IoU between two boxes [x1, y1, x2, y2]"""
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - intersection
    
    return intersection / union if union > 0 else 0.0


def collect_matched_predictions(preds, targets, iou_thresh=threshold, score_thresh=0.5):
    """
    Match predictions to ground truth and collect all predictions and labels
    for confusion matrix and classification report.
    
    Returns:
        all_pred_labels: list of predicted class labels (including background for FN)
        all_true_labels: list of ground truth class labels (including background for FP)
    """
    all_pred_labels = []
    all_true_labels = []
    
    for pred, tgt in zip(preds, targets):
        pred_boxes = pred["boxes"].cpu().numpy()
        pred_labels = pred["labels"].cpu().numpy()
        pred_scores = pred["scores"].cpu().numpy()
        
        tgt_boxes = tgt["boxes"].cpu().numpy()
        tgt_labels = tgt["labels"].cpu().numpy()
        
        # Filter predictions by score threshold
        score_mask = pred_scores >= score_thresh
        pred_boxes = pred_boxes[score_mask]
        pred_labels = pred_labels[score_mask]
        pred_scores = pred_scores[score_mask]
        
        matched_gt = set()
        matched_pred = set()
        
        # Match predictions to ground truth
        for i, pb in enumerate(pred_boxes):
            best_iou = 0.0
            best_idx = -1
            
            for j, tb in enumerate(tgt_boxes):
                if j in matched_gt:
                    continue
                iou = box_iou(pb, tb)
                if iou > best_iou:
                    best_iou = iou
                    best_idx = j
            
            if best_iou >= iou_thresh and best_idx != -1:
                # True positive or misclassification
                matched_gt.add(best_idx)
                matched_pred.add(i)
                all_pred_labels.append(pred_labels[i])
                all_true_labels.append(tgt_labels[best_idx])
            else:
                # False positive (predicted but no matching ground truth)
                # We can treat this as predicting class X when it's background (class 0)
                all_pred_labels.append(pred_labels[i])
                all_true_labels.append(0)  # Background/no object
        
        # False negatives (ground truth not matched)
        # Treat as predicting background when ground truth is class X
        for j in range(len(tgt_labels)):
            if j not in matched_gt:
                all_pred_labels.append(0)  # Predicted as background
                all_true_labels.append(tgt_labels[j])
    
    return np.array(all_pred_labels), np.array(all_true_labels)


def plot_detection_confusion_matrix(all_pred_labels, all_true_labels, class_names, 
                                   save_path='detection_confusion_matrix.png', 
                                   figsize=(12, 10), normalize=True):
    """
    Plot confusion matrix for object detection/instance segmentation
    
    Args:
        all_pred_labels: array of predicted class labels
        all_true_labels: array of ground truth class labels
        class_names: list of class names (should include 'Background' as first element)
        save_path: path to save the figure
        figsize: figure size
        normalize: whether to normalize the confusion matrix
    """
    # Create confusion matrix
    cm = confusion_matrix(all_true_labels, all_pred_labels)
    
    if normalize:
        # Normalize by row (true labels)
        cm_normalized = cm.astype('float') / (cm.sum(axis=1)[:, np.newaxis] + 1e-10)
        fmt = '.2f'
        title = 'Normalized Confusion Matrix'
    else:
        cm_normalized = cm
        fmt = 'd'
        title = 'Confusion Matrix'
    
    # Plot
    plt.figure(figsize=figsize)
    sns.heatmap(cm_normalized, annot=True, fmt=fmt, cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names, 
                cbar_kws={'label': 'Proportion' if normalize else 'Count'})
    plt.xlabel('Predicted Labels', fontsize=12)
    plt.ylabel('True Labels', fontsize=12)
    plt.title(title, fontsize=14)
    plt.tight_layout()
    plt.savefig(save_path, dpi=1200, bbox_inches='tight')
    plt.show()
    
    return cm


def print_detection_classification_report(all_pred_labels, all_true_labels, class_names):
    """
    Print classification report for object detection
    """
    # Remove background class for cleaner report (optional)
    # If you want to include background, remove these filters
    non_bg_mask = all_true_labels != 0
    filtered_true = all_true_labels[non_bg_mask]
    filtered_pred = all_pred_labels[non_bg_mask]
    
    # Also remove cases where prediction is background (optional)
    non_bg_pred_mask = filtered_pred != 0
    filtered_true = filtered_true[non_bg_pred_mask]
    filtered_pred = filtered_pred[non_bg_pred_mask]
    
    print("="*80)
    print("CLASSIFICATION REPORT (Matched Detections Only)")
    print("="*80)
    if len(filtered_true) > 0:
        report = classification_report(
            filtered_true, 
            filtered_pred, 
            labels=list(range(1, len(class_names))),  # Exclude background
            target_names=class_names[1:],  # Exclude background name
            zero_division=0
        )
        print(report)
    else:
        print("No matched detections found!")
    
    print("\n" + "="*80)
    print("CLASSIFICATION REPORT (Including False Positives and False Negatives)")
    print("="*80)
    report_full = classification_report(
        all_true_labels, 
        all_pred_labels, 
        labels=list(range(len(class_names))),
        target_names=class_names,
        zero_division=0
    )
    print(report_full)


def plot_per_class_performance(all_pred_labels, all_true_labels, class_names, 
                               save_path='per_class_performance.png'):
    """
    Plot per-class precision, recall, and F1-score as bar charts
    """
    from sklearn.metrics import precision_recall_fscore_support
    
    # Compute metrics per class
    precision, recall, f1, support = precision_recall_fscore_support(
        all_true_labels, 
        all_pred_labels, 
        labels=list(range(len(class_names))),
        zero_division=0
    )
    
    # Create subplots
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    x = np.arange(len(class_names))
    width = 0.6
    
    # Precision
    axes[0, 0].bar(x, precision, width, color='skyblue')
    axes[0, 0].set_ylabel('Precision', fontsize=12)
    axes[0, 0].set_title('Precision per Class', fontsize=14)
    axes[0, 0].set_xticks(x)
    axes[0, 0].set_xticklabels(class_names, rotation=45, ha='right')
    axes[0, 0].set_ylim([0, 1.1])
    axes[0, 0].grid(axis='y', alpha=0.3)
    
    # Recall
    axes[0, 1].bar(x, recall, width, color='lightcoral')
    axes[0, 1].set_ylabel('Recall', fontsize=12)
    axes[0, 1].set_title('Recall per Class', fontsize=14)
    axes[0, 1].set_xticks(x)
    axes[0, 1].set_xticklabels(class_names, rotation=45, ha='right')
    axes[0, 1].set_ylim([0, 1.1])
    axes[0, 1].grid(axis='y', alpha=0.3)
    
    # F1-Score
    axes[1, 0].bar(x, f1, width, color='lightgreen')
    axes[1, 0].set_ylabel('F1-Score', fontsize=12)
    axes[1, 0].set_title('F1-Score per Class', fontsize=14)
    axes[1, 0].set_xticks(x)
    axes[1, 0].set_xticklabels(class_names, rotation=45, ha='right')
    axes[1, 0].set_ylim([0, 1.1])
    axes[1, 0].grid(axis='y', alpha=0.3)
    
    # Support (number of instances)
    axes[1, 1].bar(x, support, width, color='plum')
    axes[1, 1].set_ylabel('Number of Instances', fontsize=12)
    axes[1, 1].set_title('Support per Class', fontsize=14)
    axes[1, 1].set_xticks(x)
    axes[1, 1].set_xticklabels(class_names, rotation=45, ha='right')
    axes[1, 1].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=1200, bbox_inches='tight')
    plt.show()


# ============================================================
# COMPLETE EVALUATION PIPELINE
# ============================================================

def evaluate_and_visualize(model, val_loader, device, class_names, 
                          iou_thresh=threshold, score_thresh=0.5):
    """
    Complete evaluation pipeline with confusion matrix and classification report
    
    Args:
        model: trained Mask R-CNN model
        val_loader: validation data loader
        device: torch device
        class_names: list of class names including 'Background' as first element
        iou_thresh: IoU threshold for matching boxes
        score_thresh: confidence score threshold for predictions
    """
    print("Loading model and collecting predictions...")
    model.eval()
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for images, targets in tqdm(val_loader, desc="Evaluating"):
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            
            outputs = model(images)
            
            for out, tgt in zip(outputs, targets):
                pred = {
                    "boxes": out["boxes"].cpu(),
                    "labels": out["labels"].cpu(),
                    "scores": out["scores"].cpu(),
                    "masks": out["masks"].cpu(),
                }
                gt = {
                    "boxes": tgt["boxes"].cpu(),
                    "labels": tgt["labels"].cpu(),
                    "masks": tgt["masks"].cpu(),
                }
                all_preds.append(pred)
                all_targets.append(gt)
    
    print("\nMatching predictions to ground truth...")
    all_pred_labels, all_true_labels = collect_matched_predictions(
        all_preds, all_targets, iou_thresh=iou_thresh, score_thresh=score_thresh
    )
    
    print(f"Total instances collected: {len(all_pred_labels)}")
    print(f"  - True Positives/Misclassifications: {np.sum((all_pred_labels != 0) & (all_true_labels != 0))}")
    print(f"  - False Positives: {np.sum((all_pred_labels != 0) & (all_true_labels == 0))}")
    print(f"  - False Negatives: {np.sum((all_pred_labels == 0) & (all_true_labels != 0))}")
    
    # Plot confusion matrix
    print("\nGenerating confusion matrix...")
    cm = plot_detection_confusion_matrix(
        all_pred_labels, all_true_labels, class_names,
        save_path='detection_confusion_matrix.png',
        normalize=True
    )
    
    # Plot unnormalized confusion matrix
    print("Generating unnormalized confusion matrix...")
    cm_raw = plot_detection_confusion_matrix(
        all_pred_labels, all_true_labels, class_names,
        save_path='detection_confusion_matrix_raw.png',
        normalize=False
    )
    
    # Print classification report
    print_detection_classification_report(all_pred_labels, all_true_labels, class_names)
    
    # Plot per-class performance
    print("\nGenerating per-class performance plots...")
    plot_per_class_performance(
        all_pred_labels, all_true_labels, class_names,
        save_path='per_class_performance.png'
    )
    
    return all_pred_labels, all_true_labels, cm


import torch
import numpy as np
from tqdm import tqdm

# IMPORTANT: Define your class names including 'Background' as the first element
# Example for a 4-class problem:
class_names = ['Background', 'G-cocci', 'G+cocci', 'G-bacilli', 'G+bacilli']  # Adjust to your actual classes


# Run the complete evaluation
all_pred_labels, all_true_labels, cm = evaluate_and_visualize(
    model=model,
    val_loader=val_loader,
    device=device,
    class_names=class_names,
    iou_thresh=threshold,      # IoU threshold for box matching
    score_thresh=0.5      # Confidence score threshold (adjust as needed)
)

print("\nConfusion Matrix (raw counts):")
print(cm)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import random
import torch

num_samples= 10
score_thresh=0.7

# Load the saved model state dict, put model in eval mode
best_model_path = "best_instance_segmentation.pth"
model.load_state_dict(torch.load(best_model_path))
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)
model.eval()

indices = random.sample(range(len(val_dataset)), num_samples)
fig, axes = plt.subplots(2, num_samples, figsize=(20, 5))

for idx_col, idx in enumerate(indices):
    img, target = dataset[idx]
    img_tensor = img.to(device).unsqueeze(0)

    with torch.no_grad():
        output = model(img_tensor)[0]   # predictions for this image

    # --- convert image for display ---
    img_disp = img.permute(1,2,0).cpu().numpy()
    img_disp = (img_disp - img_disp.min()) / (img_disp.max() - img_disp.min() + 1e-8)

    # === 1st row: Ground Truth ===
    axes[0, idx_col].imshow(img_disp)
    axes[0, idx_col].set_title("Ground Truth", )
    axes[0, idx_col].axis("off")

    # draw GT boxes + masks
    for box, mask, label in zip(target["boxes"], target["masks"], target["labels"]):
        x1, y1, x2, y2 = box.cpu().numpy()
        rect = patches.Rectangle((x1,y1), x2-x1, y2-y1, 
                                 linewidth=2, edgecolor=label_map[label.item()]["color_code"], facecolor="none")
        axes[0, idx_col].add_patch(rect)
        # Ground truth mask (ensure 2D)
        gt_mask = mask.squeeze().cpu().numpy()
        axes[0, idx_col].imshow(gt_mask, cmap="Reds", alpha=0.4)

    # === 2nd row: Predictions ===
    axes[1, idx_col].imshow(img_disp)
    axes[1, idx_col].set_title("Predictions")
    axes[1, idx_col].axis("off")

    scores = output["scores"].cpu().numpy()
    keep = scores >= score_thresh

    pred_boxes = output["boxes"][keep]
    pred_masks = output["masks"][keep]
    pred_labels = output["labels"][keep]

    for box, mask, label in zip(pred_boxes, pred_masks, pred_labels):
        x1, y1, x2, y2 = box.cpu().numpy()
        rect = patches.Rectangle((x1,y1), x2-x1, y2-y1, 
                                 linewidth=2, edgecolor=label_map[label.item()]["color_code"], facecolor="none")
        axes[1, idx_col].add_patch(rect)
        # Prediction mask (threshold + ensure 2D)
        pred_mask = mask.squeeze().cpu().numpy() > 0.5
        axes[1, idx_col].imshow(pred_mask, cmap="Blues", alpha=0.4)

plt.tight_layout()
plt.savefig("Prediction.png", dpi=300)
plt.show()